# ShopDesk, Module 3 Section 1 Lab 1: The CLAUDE.md Hierarchy, @import, and /memory

A beginner-friendly notebook on how Claude Code assembles project memory: a **user-level** and
**project-level** `CLAUDE.md`, an **on-demand directory-level** file, `@import` to modularize shared
standards, and moving instructions into `.claude/rules/`. Pure-Python cells make the merge, the import
resolution, and a `/memory` view concrete offline (in a **sandbox** that never touches your real
`~/.claude`); a live **Claude Agent SDK** cell loads the project memory. Runs **Sonnet**
(`claude-sonnet-4-6`) through your **Anthropic API key**.

## The real-world scenario

Your ShopDesk teammates cannot see your personal `~/.claude/CLAUDE.md`, so team conventions have to live
in the repo. A single giant `CLAUDE.md` gets unreadable, so you split shared standards into files and pull
them in with `@import`, and you keep API-specific rules near the API. When behavior looks wrong, `/memory`
tells you what actually loaded.

The question this lab answers: **how do the user, project, and directory files combine, and how do
`@import` and `.claude/rules/` keep memory modular?**

## Objectives

- Build a **user + project** `CLAUDE.md` and see how more specific scopes override broader ones.
- Use **@import** to pull shared standards into the project file.
- Move a section into **.claude/rules/** and inspect the result the way **/memory** would.

## What you'll observe

- The effective config for a file is the user, project, and (when relevant) directory layers in order.
- `@import` inlines the referenced files; imports still load at launch.
- A `/memory`-style view lists exactly which files are active for a given file.

## How to run

Run top to bottom. Building the sandbox and the pure-Python cells run anywhere and use a fake home, so
your real `~/.claude` is untouched. The live cell loads the project memory through Claude, so paste a real
key into **Setup 2/3** and re-run from the top; otherwise it skips. **Node.js 18+** is needed for the
Agent SDK.

## 0. Setup

**This cell:** installs the packages. `pyyaml` is used in the next lab; the Agent SDK drives the
live project-memory cell and needs Node.js 18+.

In [ ]:
# ===== SETUP 1/3 - install the packages =====
%pip install -q claude-agent-sdk anthropic python-dotenv pyyaml

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` for the live cell.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # filesystem paths for the sandbox
import re                                       # detect @import lines
import sys                                       # detect Windows (special event loop)
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cell will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}                                     #   carries the result/error out of the thread
    def worker():                                #   runs in its own thread
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)             #     make it this thread's loop
        try:    box["value"] = loop.run_until_complete(make_coro())   # run to completion
        except Exception as e: box["error"] = e  #     capture any error
        finally: loop.close()                    #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box: raise box["error"]        #   surface any error here
    return box.get("value")                      #   hand back the result

print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** builds a **sandbox** with a fake home and a project. We write a user-level
`CLAUDE.md`, a project `CLAUDE.md` that `@import`s two standards files, a directory-level `CLAUDE.md` for
the API folder, and one rule under `.claude/rules/`. Using a fake home means nothing here touches your real
configuration.

In [ ]:
# ===== SETUP 3/3 - build the sandbox hierarchy (fake home + project) =====
import textwrap                                    # keeps the embedded file bodies readable
SANDBOX = os.path.join(os.getcwd(), "config_sandbox")   # everything lives under here
HOME = os.path.join(SANDBOX, "home")               # a FAKE home, not your real ~
PROJECT = os.path.join(SANDBOX, "project")         # the sample project root

def write(path, content):                          # small helper: make dirs and write
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)

write(os.path.join(HOME, ".claude/CLAUDE.md"), textwrap.dedent("""\
    # User memory (personal, not shared)
    - Use 2-space indentation.
    - Explain changes briefly.
    """))
write(os.path.join(PROJECT, ".claude/CLAUDE.md"), textwrap.dedent("""\
    # ShopDesk project memory
    - Use 4-space indentation.
    @standards/style.md
    @standards/commits.md
    """))
write(os.path.join(PROJECT, ".claude/standards/style.md"), "# Style\n- Name refund helpers with a refund_ prefix.\n")
write(os.path.join(PROJECT, ".claude/standards/commits.md"), "# Commits\n- Use Conventional Commits.\n")
write(os.path.join(PROJECT, "src/api/CLAUDE.md"), "# API folder memory\n- Every endpoint validates input.\n")
write(os.path.join(PROJECT, ".claude/rules/testing.md"), "# Testing\n- Put tests under tests/ and name them test_*.py.\n")
print("sandbox at", SANDBOX)

### The load order

CLAUDE.md files combine from **broadest to most specific**, so a later layer overrides an earlier one:

- **User** `~/.claude/CLAUDE.md`: personal, loaded first, **not shared** with your team.
- **Project** `.claude/CLAUDE.md` (or root `CLAUDE.md`): shared via Git, loaded after the user file.
- **Directory** `CLAUDE.md` in a subfolder: loaded **on demand** when Claude works in that subtree.
- Personal per-project overrides go in `CLAUDE.local.md` (gitignored).

Because the user file is not shared, team conventions belong in the project file and in `.claude/rules/`.

---

### Lab objective - see how memory is assembled

**What you build:** an `@import` resolver, a layer-by-layer effective-config view, and a `/memory`-style
listing of what loaded.

**Why it helps you build real solutions:** knowing the precedence and what actually loaded is how you make
Claude follow team conventions and debug it when it does not.

**How you'll see it:** the layers stack in order, imports inline, and the memory view names every active
file.

**This cell:** an **@import** resolver. A line that is just `@path` pulls in that file's contents,
relative to the importing file. Imports can nest (up to four hops). We resolve the project `CLAUDE.md` and
show the inlined result.

In [ ]:
# ===== resolve @import directives =====
IMPORT_RE = re.compile(r"^\s*@(\S+)\s*$")         # a line that is only "@some/path"

def resolve_imports(path, depth=0):                # inline @imports, relative to each file's folder
    if depth > 4:                                   #   imports nest at most four hops deep
        return "[import depth exceeded]"
    base = os.path.dirname(path)                     #   imports are relative to the importing file
    lines = []
    for line in open(path).read().splitlines():
        m = IMPORT_RE.match(line)
        if m:
            lines.append(resolve_imports(os.path.join(base, m.group(1)), depth + 1))   # inline it
        else:
            lines.append(line)
    return "\n".join(lines)

print(resolve_imports(os.path.join(PROJECT, ".claude/CLAUDE.md")))

**This cell:** note what `@import` does and does not do. It **organizes** your instructions into
separate files, but the imported files still load at launch, so it does **not** reduce context cost. The
mechanism that saves context is path-scoped `.claude/rules/`, which is the next lab.

In [ ]:
# ===== imports organize, but still load at launch =====
resolved = resolve_imports(os.path.join(PROJECT, ".claude/CLAUDE.md"))   # the inlined project memory
imported_files = [".claude/standards/style.md", ".claude/standards/commits.md"]           # what got pulled in
print("imported files still loaded at launch:", imported_files)
print("project memory size after inlining (chars):", len(resolved))

**This cell:** the **effective config** for a given file. We stack the layers broadest to most
specific: user, then project, then the directory file when the target is under `src/api/`. The later layer
wins on conflicts, so the project's 4-space rule overrides the user's 2-space rule inside the project.

In [ ]:
# ===== assemble the layers for a target file =====
def effective_config(target):                      # target path -> ordered (scope, source) layers
    layers = [("user", os.path.join(HOME, ".claude/CLAUDE.md")),
              ("project", os.path.join(PROJECT, ".claude/CLAUDE.md"))]
    if target.startswith("src/api/"):               #   directory memory loads on demand for this subtree
        layers.append(("dir:src/api", os.path.join(PROJECT, "src/api/CLAUDE.md")))
    return layers

for target in ["src/api/orders.py", "src/util/math.py"]:   # one inside the API folder, one outside
    scopes = [scope for scope, _ in effective_config(target)]
    print(f"  {target:20} -> {scopes}")

**This cell:** resolve a **conflict**. Both the user and project files set an indentation rule. We
read the winning value by walking the layers in order and letting the most specific one override, which is
exactly how Claude treats overlapping instructions.

In [ ]:
# ===== more specific overrides broader =====
def indent_rule(target):                           # walk layers; last matching value wins
    winner = None
    for scope, src in effective_config(target):
        for line in resolve_imports(src).splitlines() if scope == "project" else open(src).read().splitlines():
            m = re.search(r"Use (\d+)-space indentation", line)
            if m:
                winner = (scope, m.group(1))         #   overwrite as we go broad -> specific
    return winner

print("indentation for a project file:", indent_rule("src/api/orders.py"))   # project (4) overrides user (2)

**This cell:** a `/memory`-style view. In the real CLI, `/memory` shows what is loaded and lets you
edit it. Here we list the active files for a target in load order, marking the directory file as
conditional so you can see why a rule did or did not apply.

In [ ]:
# ===== simulate the /memory listing =====
def memory_view(target):                           # print the loaded files for a target, in order
    print(f"/memory for {target}:")
    for scope, src in effective_config(target):
        tag = " (on demand)" if scope.startswith("dir") else ""
        print(f"  [{scope}]{tag} {os.path.relpath(src, SANDBOX)}")
    print("  [rules] project/.claude/rules/testing.md (loaded as project memory)")

memory_view("src/api/orders.py")
print()
memory_view("src/util/math.py")

**This cell:** **move a section into a rule**. Suppose the project `CLAUDE.md` had a Testing section;
extracting it into `.claude/rules/testing.md` keeps the main file lean. We compare line counts before and
after. Note: without a `paths` field the rule still loads at launch, so this improves readability now, and
the next lab adds path scoping to also cut context.

In [ ]:
# ===== extract a section into .claude/rules/ =====
monolithic = resolve_imports(os.path.join(PROJECT, ".claude/CLAUDE.md")) + "\n# Testing\n- Put tests under tests/.\n- Name them test_*.py.\n"
slim = resolve_imports(os.path.join(PROJECT, ".claude/CLAUDE.md"))          # Testing now lives in rules/
print("monolithic CLAUDE.md lines:", len(monolithic.splitlines()))
print("slim CLAUDE.md lines       :", len(slim.splitlines()), "(Testing moved to .claude/rules/testing.md)")

**This cell:** the live view. We point the Agent SDK at the project with `cwd=PROJECT` and
`setting_sources=["project"]`, which loads the project `CLAUDE.md` and its rules, then ask a question the
project memory answers. This shows project memory reaching the model. The live cell loads **project** memory
only; the user layer is shown offline above so we never read your real home.

In [ ]:
# ===== live: load the project memory and ask =====
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock

MEM_OPTS = ClaudeAgentOptions(                     # load project CLAUDE.md and rules from the sandbox
    model=MODEL, cwd=PROJECT,
    setting_sources=["project"],                    # this is what loads project memory
    allowed_tools=["Read"])                         # read-only is enough here

async def ask(prompt):                              # stream just the text answer
    async for m in query(prompt=prompt, options=MEM_OPTS):
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, TextBlock) and b.text.strip(): print("  ", b.text.strip()[:160])

if RUN_LIVE:                                        # needs a real key (and Node.js 18+)
    run_async(lambda: ask("Based on this project's conventions, what indentation should I use? One line."))
else:
    print("[skipped] expected: Claude answers 4-space, reflecting the project CLAUDE.md.")

**In the real Claude Code CLI** (reference, not run here):

```text
/memory      # show and edit what is loaded (user, project, rules, auto memory)
/init        # generate a starter CLAUDE.md for a project
```

Keep each `CLAUDE.md` under about 200 lines; longer files dilute adherence. Put team conventions in the
project file and `.claude/rules/`, and personal per-project overrides in `CLAUDE.local.md` (gitignored).

| anti-pattern | what to do instead |
|---|---|
| put team conventions in `~/.claude/CLAUDE.md` | it is not shared; use the project file and rules |
| one giant `CLAUDE.md` | split with `@import` and move topics into `.claude/rules/` |
| assume a rule loaded | check with `/memory` |
| rely on `@import` to cut context | imports still load at launch; use path-scoped rules |

**Lesson:** project memory is layered: user, then project, then directory, with the most specific
winning. Because the user file is not shared, team conventions live in the project file and in
`.claude/rules/`. `@import` keeps files modular but does not save context, and `/memory` is how you see what
actually loaded.

---

## Recap - the configuration hierarchy

| Layer | File | Scope |
|---|---|---|
| User | `~/.claude/CLAUDE.md` | personal, not shared, loaded first |
| Project | `.claude/CLAUDE.md` | shared via Git, overrides user |
| Directory | subfolder `CLAUDE.md` | on demand, overrides project there |
| Rules | `.claude/rules/*.md` | project memory; path-scoping comes next |

One principle to carry forward: **put shared conventions where the team gets them, and verify with
/memory.** To run live, paste a real key into **Setup 2/3** and re-run from the top. Then try it: add a
directory `CLAUDE.md` under `src/` and watch which files it applies to. Next lab: path-scoped rules and the
context they save.